<a href="https://colab.research.google.com/github/honeykgarg/PyTorch_NN/blob/regression_with_regularization/LinearRegressionWithPyTorchNNLayers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install hiddenlayer

In [ ]:
import torch
import hiddenlayer as hl
import pandas as pd

import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

import sklearn

In [ ]:
!pip install --upgrade kagglehub

In [ ]:
import kagglehub
# You may need to re-run this cell after logging in.
kagglehub.login()


In [ ]:
path = kagglehub.competition_download('bike-sharing-demand')

In [ ]:
path

In [ ]:
import os

dataset_path = '/root/.cache/kagglehub/competitions/bike-sharing-demand'
print(f"Contents of {dataset_path}:")
!ls -lh {dataset_path}

In [ ]:
data_path = os.path.join(dataset_path, 'train.csv')
data = pd.read_csv(data_path)

print("Sample Submission DataFrame Head:")
print(data.head())

print("\nSample Submission DataFrame Info:")
data.info()


data.shape



In [ ]:
# Convert 'datetime' column to datetime objects
data['datetime'] = pd.to_datetime(data['datetime'])

# Get the minimum and maximum datetime
min_datetime = data['datetime'].min()
max_datetime = data['datetime'].max()

print(f"Minimum datetime: {min_datetime}")
print(f"Maximum datetime: {max_datetime}")

In [ ]:
data['year'] = data['datetime'].dt.year
data.head()

In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(x='temp', y='count', hue='season', data=data, errorbar=None)

plt.legend(loc = 'upper right', bbox_to_anchor= (1.2,0.5))

plt.xlabel('Year')

plt.ylabel('Total number of bikes rented')

plt.title('Number of bikes renter per season')

In [ ]:
season_mapping = {
    1: 'Spring',
    2: 'Summer',
    3: 'Fall',
    4: 'Winter' # Assuming 4 is Winter based on common season numbering
}
data['seasonName'] = data['season'].map(season_mapping)

print(data[['season', 'seasonName']].head())

In [ ]:
data.info()

In [ ]:
dataWithSeasonHotEncoded = pd.get_dummies(data, columns=['seasonName'], dtype=int)

dataWithSeasonHotEncoded.head()

In [ ]:
columns = ['registered', 'holiday','workingday','weather', 'temp','atemp','seasonName_Spring','seasonName_Summer','seasonName_Fall','seasonName_Winter']

features = dataWithSeasonHotEncoded[columns]

features.iloc[10347]

In [ ]:
target = dataWithSeasonHotEncoded[['count']]

In [ ]:
features.iloc[10347]

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.2)

In [ ]:
x_train.head()
y_test.iloc[1277]

In [ ]:
x_test.head()

In [ ]:
x_train_tensor = torch.from_numpy(x_train.values.astype(float)).to(device)
x_test_tensor = torch.from_numpy(x_test.values.astype(float)).to(device)
y_train_tensor = torch.from_numpy(y_train.values.astype(float)).to(device)
y_test_tensor = torch.from_numpy(y_test.values.astype(float)).to(device)

In [ ]:
x_train_tensor.shape

In [ ]:
import torch.utils.data as data_utils

In [ ]:
train_data = data_utils.TensorDataset(x_train_tensor, y_train_tensor)
train_data

In [ ]:
train_loader = data_utils.DataLoader(train_data, batch_size=100, shuffle=True)

In [ ]:
len(train_loader)

In [ ]:
features_batch, target_batch = next(iter(train_loader))

In [ ]:
features_batch.shape

In [ ]:
inp = x_train_tensor.shape[1]

In [ ]:
out  = 1

In [ ]:
hid  = 10

In [ ]:
loss_fn = torch.nn.MSELoss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
model = torch.nn.Sequential(torch.nn.Linear(inp,hid), torch.nn.ReLU(), torch.nn.Linear(hid, out)).to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
total_step = len(train_loader)
y_train.head()
epochs = 5000
for epoch in range(epochs + 1):
    for i, (features_batch, target_batch) in enumerate(train_loader):

        # Forward pass
        output = model(features_batch.float())

        # Calculate the loss
        loss = loss_fn(output, target_batch.float())

        # Zero the gradients
        optimizer.zero_grad()

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

    if (epoch) % 2 == 0:
        print('Epoch [{}/{}], Step[{}/{}], Loss: {:.4f}'.format(epoch+1, epochs, i+1, total_step, loss.item()))

print("Training complete!")

In [ ]:
model.eval()

with torch.no_grad():
    y_pred = model(x_test_tensor.float())

In [ ]:
sample = x_test.iloc[45]
sample

In [ ]:
sample_tensor = torch.tensor(sample.values.astype(float)).to(device)
sample_tensor

In [ ]:
with torch.no_grad():
  y_pred = model(sample_tensor.float())

print("Predicted count:", (y_pred.item()))
print("Actual count:", (y_test.iloc[45]))

In [ ]:
with torch.no_grad():
    y_pred = model(x_test_tensor.float())

In [ ]:
y_pred_np = y_pred.detach().cpu().numpy()

y_pred_np.shape

In [ ]:
y_test.values.shape

In [ ]:
compare_df = pd.DataFrame({'actual' : np.squeeze(y_test.values), 'predicted' : np.squeeze(y_pred.cpu().numpy())})
compare_df.sample(20)

In [ ]:
sklearn.metrics.r2_score(y_test, y_pred.cpu())